У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTENC
from imblearn.combine import SMOTETomek
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.multiclass import OneVsRestClassifier

In [33]:
df = pd.read_csv('/content/drive/MyDrive/ML_homework/customer_segmentation_train.csv')

In [34]:
df.head()

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


In [36]:
input_cols = df.drop(['ID', 'Segmentation'], axis=1).columns.to_list()
target_col = 'Segmentation'

In [37]:
input_cols, target_col

(['Gender',
  'Ever_Married',
  'Age',
  'Graduated',
  'Profession',
  'Work_Experience',
  'Spending_Score',
  'Family_Size',
  'Var_1'],
 'Segmentation')

In [38]:
train_inputs, test_inputs, train_targets, test_targets  = train_test_split(df[input_cols], df[target_col], test_size=0.2, random_state=17, stratify=df[target_col])

In [39]:
print('train_inputs.shape :', train_inputs.shape)
print('train_targets.shape :', train_targets.shape)
print('test_inputs.shape :', test_inputs.shape)
print('test_targets.shape :', test_targets.shape)

train_inputs.shape : (6454, 9)
train_targets.shape : (6454,)
test_inputs.shape : (1614, 9)
test_targets.shape : (1614,)


In [40]:
numeric_cols = train_inputs.select_dtypes('number').columns.to_list()
categorical_cols = train_inputs.select_dtypes('object').columns.to_list()

In [41]:
numeric_cols, categorical_cols

(['Age', 'Work_Experience', 'Family_Size'],
 ['Gender',
  'Ever_Married',
  'Graduated',
  'Profession',
  'Spending_Score',
  'Var_1'])

In [42]:
train_inputs[numeric_cols].describe().round(2)

,Age,Work_Experience,Family_Size
count,6454.00,5785.00,6189.00
mean,43.27,2.63,2.86
std,16.65,3.39,1.53
min,18.00,0.00,1.00
25%,30.00,0.00,2.00
50%,40.00,1.00,3.00
75%,53.00,4.00,4.00
max,89.00,14.00,9.00


In [43]:
numeric_imputer = SimpleImputer(strategy = 'median')

numeric_imputer.fit(train_inputs[numeric_cols])

train_inputs[numeric_cols] = numeric_imputer.transform(train_inputs[numeric_cols])
test_inputs[numeric_cols] = numeric_imputer.transform(test_inputs[numeric_cols])

In [44]:
categorical_imputer = SimpleImputer(strategy='most_frequent')

categorical_imputer.fit(train_inputs[categorical_cols])

train_inputs[categorical_cols] = categorical_imputer.transform(train_inputs[categorical_cols])
test_inputs[categorical_cols] = categorical_imputer.transform(test_inputs[categorical_cols])

In [45]:
scaler = MinMaxScaler()

scaler.fit(train_inputs[numeric_cols])

train_inputs[numeric_cols] = scaler.transform(train_inputs[numeric_cols])
test_inputs[numeric_cols] = scaler.transform(test_inputs[numeric_cols])

In [46]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoder.fit(train_inputs[categorical_cols])

encoded_cols = list(encoder.get_feature_names_out(categorical_cols))

train_inputs[encoded_cols] = encoder.transform(train_inputs[categorical_cols])
test_inputs[encoded_cols] = encoder.transform(test_inputs[categorical_cols])

In [47]:
X_train = train_inputs[numeric_cols + encoded_cols]
X_test = test_inputs[numeric_cols + encoded_cols]

In [48]:
log_reg = LogisticRegression(solver='liblinear')

In [49]:
log_reg.fit(X_train, train_targets)

LogisticRegression(solver='liblinear')

In [50]:
y_pred = log_reg.predict(X_test)

**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [51]:
encoded_cols = list(encoder.get_feature_names_out(categorical_cols))
print(encoded_cols)

['Gender_Female', 'Gender_Male', 'Ever_Married_No', 'Ever_Married_Yes', 'Graduated_No', 'Graduated_Yes', 'Profession_Artist', 'Profession_Doctor', 'Profession_Engineer', 'Profession_Entertainment', 'Profession_Executive', 'Profession_Healthcare', 'Profession_Homemaker', 'Profession_Lawyer', 'Profession_Marketing', 'Spending_Score_Average', 'Spending_Score_High', 'Spending_Score_Low', 'Var_1_Cat_1', 'Var_1_Cat_2', 'Var_1_Cat_3', 'Var_1_Cat_4', 'Var_1_Cat_5', 'Var_1_Cat_6', 'Var_1_Cat_7']


In [52]:
categorical_cols_index = [X_train.columns.get_loc(col) for col in encoded_cols]

In [53]:
smotenc = SMOTENC(categorical_features=categorical_cols_index, random_state=17)
X_train_smotenc, y_train_smotenc = smotenc.fit_resample(X_train, train_targets)

In [54]:
model_smotenc = LogisticRegression(solver='liblinear')

In [55]:
model_smotenc.fit(X_train_smotenc, y_train_smotenc)

LogisticRegression(solver='liblinear')

In [56]:
y_pred_smotenc = model_smotenc.predict(X_test)

In [57]:
smotenc_tomek = SMOTETomek(smote=smotenc, random_state=17)
X_train_smotenc_tomek, y_train_smotenc_tomek = smotenc_tomek.fit_resample(X_train, train_targets)

In [58]:
model_smotenc_tomek = LogisticRegression(solver='liblinear')

In [59]:
model_smotenc_tomek.fit(X_train_smotenc_tomek, y_train_smotenc_tomek)

LogisticRegression(solver='liblinear')

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [60]:
model_orig = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
model_orig.fit(X_train, train_targets)
y_pred_orig = model_orig.predict(X_test)

print(classification_report(test_targets, y_pred_orig))

              precision    recall  f1-score   support

           A       0.40      0.51      0.45       394
           B       0.37      0.15      0.21       372
           C       0.50      0.64      0.56       394
           D       0.63      0.64      0.63       454

    accuracy                           0.50      1614
   macro avg       0.48      0.48      0.46      1614
weighted avg       0.48      0.50      0.47      1614



In [61]:
model_smotenc = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
model_smotenc.fit(X_train_smotenc, y_train_smotenc)
y_pred_smotenc = model_smotenc.predict(X_test)

print(classification_report(test_targets, y_pred_smotenc))

              precision    recall  f1-score   support

           A       0.40      0.53      0.46       394
           B       0.39      0.21      0.27       372
           C       0.51      0.62      0.56       394
           D       0.64      0.60      0.62       454

    accuracy                           0.50      1614
   macro avg       0.49      0.49      0.48      1614
weighted avg       0.49      0.50      0.49      1614



In [62]:
model_smotenc_tomek = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
model_smotenc_tomek.fit(X_train_smotenc_tomek, y_train_smotenc_tomek)
y_pred_smotenc_tomek = model_smotenc_tomek.predict(X_test)

print(classification_report(test_targets, y_pred_smotenc_tomek))

              precision    recall  f1-score   support

           A       0.40      0.53      0.45       394
           B       0.36      0.20      0.26       372
           C       0.52      0.60      0.56       394
           D       0.65      0.60      0.62       454

    accuracy                           0.49      1614
   macro avg       0.48      0.48      0.47      1614
weighted avg       0.49      0.49      0.48      1614



**Спостереження**

Метрики "macro avg" та "weighted avg" дуже близькі для всіх трьох моделей, але у варіанті з SMOTENC все ж найкращі показники